# ESPRIT and Gridless Estimation

In [ ]:
# Importing modules
import numpy as np;
import matplotlib.pyplot as plt; 

In [ ]:
# Defining methods
hermitian = lambda array: np.conj(array).T


def steering_vector(
    sensor_pos: np.ndarray,
    ang_elev: float,
    ang_azim: float,
    wavelen: float
    ) -> np.ndarray:
  iota = np.array([[np.sin(ang_elev)*np.cos(ang_azim)],
                    [np.sin(ang_elev)*np.sin(ang_azim)],
                    [np.cos(ang_elev)]]);
  return np.exp(1j*2*np.pi/wavelen*sensor_pos.T@iota); 


def d_steering_vector(
    sensor_pos: np.ndarray,
    ang_elev: float,
    ang_azim: float,
    wavelen: float
    ) -> np.ndarray:
  a = steering_vector(sensor_pos, ang_elev, ang_azim, wavelen);
  d_omega = np.array([[np.cos(ang_elev)*np.cos(ang_azim)],
                      [np.cos(ang_elev)*np.sin(ang_azim)],
                      [-np.sin(ang_elev)]]);
  return -1j*2*np.pi/wavelen * a * sensor_pos.T@d_omega; 


def generate_pos_1d_ula(
    N: int,
    d: float,
    axis=(1.,0.,0.),
    x_init=(0.,0.,0.)
    ) -> np.ndarray:
  if len(axis) != 3:
    raise TypeError(f"""The axis argument represents a 3D Cartesian vector.
      The length of the input ({len(axis)}) does not match with expected
      size 3.""")
  if sum(axis) != 1:
    axis_new = (x/sum(axis) for x in axis); 
    axis = axis_new; 

  if len(x_init) != 3:
    raise TypeError(f"""The x_init argument represents a 3D cartesian vector.
      The length of the input ({len(x_init)}) does not match with expected
      size 3."""); 

  sensor_pos = np.tile(x_init, N).reshape(N,3).T \
    + (np.arange(0,N,1)*d) * np.tile(np.array(axis), N).reshape(N,3).T; 
  return sensor_pos; 


def calculate_crb(sensor_pos, N, angs_elev, snr_db, S_db):
  noise_pow = 10**(-snr_db/10); 
  sig_pow = [10**(s_db)/10 for s_db in S_db]; 
  R_ss = np.diag(np.array(sig_pow)); 

  K = angs_elev.shape[0]; 

  # Steering matrix (A_mat) Derivative matrix (D_mat) of steering vectors w.r.t angles (in degrees)
  A_mat = np.zeros((N, K), dtype=complex); 
  D_mat = np.zeros((N, K), dtype=complex); 

  for i, theta in enumerate(angs_elev):
    A_mat[:,i] = steering_vector(sensor_pos, np.deg2rad(theta), np.deg2rad(0), wl)[:,0]; 
    D_mat[:,i] = d_steering_vector(sensor_pos, np.deg2rad(theta), np.deg2rad(0), wl)[:,0]; 

  Rxx_true = A_mat@R_ss@A_mat.conj().T + noise_pow*np.eye(N); 
  Rxx_inv = np.linalg.inv(Rxx_true); 

  P_A_perp = np.eye(N) - A_mat@np.linalg.inv(A_mat.conj().T@A_mat)@A_mat.conj().T; # Projection matrix onto the noise subspace

  # The Fisher Information Matrix components
  term1 = D_mat.conj().T @ P_A_perp @ D_mat; 
  term2 = (R_ss @ A_mat.conj().T @ Rxx_inv @ A_mat @ R_ss).T; 

  FIM = (2*T/noise_pow) * np.real(term1 * term2); 

  return np.linalg.inv(FIM); 


def generate_random_targets(ang_min: float, ang_max: float, ang_dist: float, K: int) -> np.ndarray:
  while True:
    angs = np.random.uniform(ang_min, ang_max, K, dtype=np.float64); 
    angs_sort = np.sort(angs); 
    for i in range(K-1):  # break for if is not valid
      is_valid = np.abs(angs_sort[i+1] - angs_sort[i]) >= ang_dist; 
      if not is_valid:
        break; 
    if is_valid: break;   # break while if is valid
  return angs; 

In [ ]:
# Benchmark methods
def doa_est_bartlett(
    theta_scan: np.ndarray,
    sensor_pos: np.ndarray,
    R_xx: np.ndarray,
    wl: float) -> np.ndarray:
  P_bartlett = np.zeros(len(theta_scan), dtype=np.complex128); 

  for i, theta in enumerate(theta_scan):
    bartlett = steering_vector(sensor_pos, np.deg2rad(theta), np.deg2rad(0), wl); 
    P_bartlett[i] = np.squeeze(hermitian(bartlett) @ R_xx @ bartlett); 

  return P_bartlett; 


def doa_est_capon(
    theta_scan: np.ndarray,
    sensor_pos: np.ndarray,
    R_xx: np.ndarray,
    wl: float) -> np.ndarray:
  Rxx_inv = np.linalg.inv(R_xx); 
  P_capon = np.zeros(len(theta_scan), dtype=np.complex128); 

  for i, theta in enumerate(theta_scan):
    bartlett = steering_vector(sensor_pos, np.deg2rad(theta), np.deg2rad(0), wl); 
    capon = (Rxx_inv @ bartlett) / (hermitian(bartlett) @ Rxx_inv @ bartlett); 
    P_capon[i] = np.squeeze(hermitian(capon) @ R_xx @ capon); 

  return P_capon; 


def doa_est_music(theta_scan, sensor_pos, R_xx, wl, K):
  _, e_vec = np.linalg.eigh(R_xx); 
  Un = e_vec[:, :-K]; 

  P_music = np.zeros(len(theta_scan), dtype=np.float64); 

  for i, theta in enumerate(theta_scan):
      a_theta = steering_vector(sensor_pos, np.deg2rad(theta), np.deg2rad(0), wl); 
      denominator = np.abs(a_theta.conj().T @ Un @ Un.conj().T @ a_theta); 
      P_music[i] = 1 / np.squeeze(denominator); 

  return P_music; 

## Parameters

* $c$: Speed of light. In this example, assumed to be rounded to $30000000$ m/s.
* $f$: Frequency of the target signals (in Hz).
* $\lambda$: Wavelength of the target signals (in m).

* $N$: # of antenna elements
* $T$: # of snapshots
* $K$: # of targets

In [ ]:
# Defining parameters
c = 3*1e8;      # The speed of light in vacuum (m/s)
f = 5*1e9;      # Narrowband signal frequency (Hz)
wl = c/f;       # Signal wavelenght (m)
d = wl/2;       # Antenna distance (m)

N = 16;         # Number of antennas
T = 1000;       # Number of snapshots
K = 3;          # Number of targets

S_dB = [0]*K;   # Target signal power (dB)
SNR_dB = 10.0;  # Signal-to-Noise Ratio (dB)

ang_azim = 60;  # Azimuth angle (deg)
ang_min = -60;  # Minimum elevation angle (deg)
ang_max = 90;   # Maximum elevation angle (deg)
ang_dist = 5;   # Minimum distance between targets (deg)
ang_res = 0.1   # Angular resolution in scanning (deg)

sensor_pos = generate_pos_1d_ula(N, d);                                 # Sensor position
theta_scan = np.arange(-90,90+ang_res,ang_res);                         # Angle scan
true_angles = generate_random_targets(ang_min, ang_max, ang_dist, K);   # Target elevation angles

for k in range(K):
  print(f"Target {k} True Angle: {true_angles[k]:.3f}°"); 

In [ ]:
# Target Angle Generation
while True:
  true_angles = np.random.uniform(ang_min, ang_max, K); 
  angles_sort = np.sort(true_angles)
  for i in range(K-1): # break for if is not valid
    is_valid = np.abs(angles_sort[i+1] - angles_sort[i]) >= ang_dist; 
    if not is_valid:
      break; 
  if is_valid: break; # break while if is valid

for k in range(K):
  print(f"Target {k} True Angle: {true_angles[k]:.3f}°"); 

Target 0 True Angle: 83.166°
Target 1 True Angle: 55.263°
Target 2 True Angle: -27.898°
Target 3 True Angle: 47.609°
Target 4 True Angle: 21.491°


In [ ]:
A = np.column_stack([steering_vector(sensor_pos, np.deg2rad(theta), np.deg2rad(0), wl) for theta in true_angles]); # Array Steering Matrix

# Data Generation
S = (np.random.randn(K, T) + 1j*np.random.randn(K, T))/np.sqrt(2); # Target Signals: Uncorrelated Gaussian (variance = 1.0)

# Noise: Spatially white complex Gaussian noise
noise_var = 10**(-SNR_dB/10); 
Noise = np.sqrt(noise_var) * (np.random.randn(N, T) + 1j*np.random.randn(N, T))/np.sqrt(2); 

# Received Signal at the Array
X = A @ S + Noise;

## ESPRIT Algorithm

Paper for the ESPRIT algorithm is [here](https://ieeexplore.ieee.org/abstract/document/32276).

In [ ]:
# The ESPRIT algorithm
R_xx = (X @ X.conj().T)/T;      # Sample covariance matrix
U, _, _ = np.linalg.svd(R_xx);  # Left singular matrix
Us = U[:, :K];                  # Signal Subspace

# Maximally overlapping Subarrays (N-1)
U1 = Us[:-1, :];                # First M-1 rows (Subarray 1)
U2 = Us[1:, :];                 # Last M-1 rows (Subarray 2)

# Total Least Squares (TLS) to find Rotation Matrix (Psi)
U12 = np.hstack((U1, U2));

_, _, Vh_tls = np.linalg.svd(U12); 
V_tls = Vh_tls.T.conj();        # Get right singular vectors as columns

# Partition the V matrix to isolate the null space (right half of V)
V12 = V_tls[:K, K:];            # Top-right K x K block
V22 = V_tls[K:, K:];            # Bottom-right K x K block

# Calculate the TLS Rotation Matrix
Psi_tls = -V12 @ np.linalg.inv(V22); 

eigenvalues = np.linalg.eigvals(Psi_tls); # Complex eigenvalues of the rotation matrix
phases = np.angle(eigenvalues);           # The phases (angles) of the eigenvalues

# Back-calculate the spatial angles of arrival (in radians, then to degrees)
estimated_angles_rad = np.arcsin(phases / (2 * np.pi * (d / wl))); 
estimated_angles_deg = np.sort(np.rad2deg(estimated_angles_rad)); 

In [ ]:
# Printing Results
true_angles_sorted = true_angles[np.argsort(true_angles)]; 
S_dB_sorted = np.array(S_dB)[np.argsort(true_angles)]; 

## Cramer-Rao Bound
CRB_mat = calculate_crb(sensor_pos, N, true_angles_sorted, SNR_dB, S_dB_sorted); 
CRB_var = np.diag(CRB_mat);       # The diagonal elements are the minimum variances for target 1 and target 2
CRB_rmse = np.sqrt(CRB_var);      # Convert variance (deg^2) to Standard Deviation / RMSE (deg)

print(f"--- DoA Estimation with the ESPRIT Algorithm ---"); 
print(f"Array Size:        {N} elements"); 
print(f"Snapshots:         {T}"); 
print(f"Target No.:        {K}"); 
print(f"SNR:               {SNR_dB} dB\n"); 

print("Target | True Angle (deg) | Est. Angle (deg) | Error (deg) | CRLB (deg) |"); 
print("-" * 72);
for k in range(K):
    error = np.abs(true_angles_sorted[k] - estimated_angles_deg[k]); 
    print(f"  {k+1}    | {true_angles_sorted[k]:>16.4f} | {estimated_angles_deg[k]:>16.4f} | {error:>11.4f} | {CRB_rmse[k]:>10.4f} |"); 

--- DoA Estimation with the ESPRIT Algorithm ---
Array Size:        16 elements
Snapshots:         100
Target No.:        5
SNR:               10.0 dB

Target | True Angle (deg) | Est. Angle (deg) | Error (deg) | CRLB (deg) |
------------------------------------------------------------------------
  1    |         -27.8978 |         -27.8456 |      0.0523 |     0.0014 |
  2    |          21.4906 |          21.4775 |      0.0131 |     0.0014 |
  3    |          47.6094 |          47.4728 |      0.1366 |     0.0035 |
  4    |          55.2626 |          55.2824 |      0.0198 |     0.0044 |
  5    |          83.1657 |          83.2052 |      0.0395 |     0.0116 |


In [ ]:
# Running the compared algorithms (DAS, MVDR, and MUSIC)
Rxx_samp = (X @ X.conj().T)/T; 

P_bartlett = doa_est_bartlett(theta_scan, sensor_pos, Rxx_samp, wl); 
P_bartlett_dB = 10*np.log10(np.abs(np.multiply(P_bartlett/np.amax(P_bartlett), np.conjugate(P_bartlett/np.amax(P_bartlett))))); 

P_capon = doa_est_capon(theta_scan, sensor_pos, Rxx_samp, wl); 
P_capon_dB = 10*np.log10(np.abs(np.multiply(P_capon/np.amax(P_capon), np.conjugate(P_capon/np.amax(P_capon))))); 

P_music = doa_est_music(theta_scan, sensor_pos, Rxx_samp, wl, K); 
P_music_dB = 10 * np.log10(P_music / np.max(P_music)); 

In [ ]:
# Visualization
plt.rcParams.update({'font.size': 12}); 

plt.figure(figsize=(10, 6));
plt.plot(theta_scan, P_music_dB, linewidth=2, color='blue', label='MUSIC'); 
plt.plot(theta_scan, P_capon_dB, linewidth=2, color='orange', label='MVDR'); 
plt.plot(theta_scan, P_bartlett_dB, linewidth=2, color='magenta', label='MVDR'); 

for i, angle in enumerate(true_angles):
    plt.axvline(x=angle, color='red', linestyle='--',
                label=f'True Angle ({angle:.2f}°) ±{CRB_rmse[i]:.4f}°'); 

plt.title(f'MUSIC DoA Estimation (N={N}, K={K}, T={T}, SNR={snr_db}dB)'); 
plt.xlabel('Angle (deg)'); 
plt.ylabel('Spatial Pseudo-spectrum (dB)'); 
plt.xlim([ang_min, ang_max]); 
plt.ylim([-40, 5]);
plt.grid(True, linestyle=':', alpha=0.7); 

handles, labels = plt.gca().get_legend_handles_labels(); 
by_label = dict(zip(labels, handles)); 
plt.legend(by_label.values(), by_label.keys()); 

plt.tight_layout(); 
plt.show(); 